# A Full Training Loop

In this section, we will implement a full training loop from scratch using PyTorch. This gives us more control over the training process compared to using the `Trainer` class. First, let's install the required libraries if they aren't already installed.

In [ ]:
#!pip install datasets evaluate transformers[sentencepiece]
#!pip install accelerate

## 1. Preparing the Data

First, we need to load our dataset (MRPC from GLUE) and initialize our tokenizer. We define a `tokenize_function` to tokenize sentence pairs and apply it across our entire dataset using the `map` function. Finally, we set up a `DataCollatorWithPadding` to handle dynamic padding of batches.

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

## 2. Post-processing the Tokenized Datasets

Before we can create PyTorch DataLoaders, we need to perform some post-processing on our tokenized datasets to match what PyTorch expects:
1. **Remove unnecessary columns**: We remove `sentence1`, `sentence2`, and `idx` because the model doesn't expect them.
2. **Rename the label column**: We rename `label` to `labels` to match the argument name the model expects.
3. **Set the format**: We set the dataset format to return PyTorch tensors (`torch`) instead of standard lists.

In [3]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

## 3. Creating DataLoaders

With our datasets properly formatted as PyTorch tensors, we can now create `DataLoader` instances. We use `shuffle=True` for the training set to ensure the model doesn't learn the order of the data. The `collate_fn` handles the dynamic padding for each batch.

In [4]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

Let's inspect the first batch from our training dataloader to ensure everything is shaped correctly. The shapes should match our batch size (8) and the maximum sequence length in this specific batch.

In [5]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 69]),
 'token_type_ids': torch.Size([8, 69]),
 'attention_mask': torch.Size([8, 69])}

## 4. Initializing the Model

We initialize our sequence classification model using `AutoModelForSequenceClassification`. Since the MRPC dataset has two classes (paraphrase or not), we set `num_labels=2`. Note that we might see warnings about newly initialized weights for the classification head, which is expected since we are fine-tuning.

In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Let's do a quick sanity check by passing our first batch through the model. It should output the loss and the logits (with shape [8, 2]).

In [7]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(0.6188, grad_fn=<NllLossBackward0>) torch.Size([8, 2])


## 5. Setting up the Optimizer

We use the `AdamW` optimizer, which is standard for transformer models. It's similar to Adam but implements decoupled weight decay regularization which works better for these architectures. We set a learning rate of `5e-5`.

In [8]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

## 6. Setting up the Learning Rate Scheduler

We define a linear learning rate scheduler that decays the learning rate linearly from its initial value to 0 over the total number of training steps. The total number of training steps is the number of epochs (3) multiplied by the number of batches in our training dataloader.

In [9]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)

1377


## 7. Device Preparation

To speed up training, we want to run our model on a GPU if one is available. We define a PyTorch device (`cuda` or `cpu`) and move our model to that device.

In [10]:
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device

device(type='cuda')

## 8. The Training Loop

Now we write the actual training loop. For each epoch, we iterate through the training dataloader:
1. Move the batch to the device (GPU/CPU).
2. Perform a forward pass (`model(**batch)`).
3. Calculate the loss.
4. Perform a backward pass (`loss.backward()`) to calculate gradients.
5. Update the model weights (`optimizer.step()`).
6. Update the learning rate (`lr_scheduler.step()`).
7. Clear the gradients (`optimizer.zero_grad()`) for the next step.

We also use `tqdm` to display a progress bar.

In [11]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

  0%|          | 0/1377 [00:00<?, ?it/s]

## 9. The Evaluation Loop

After training, we need to evaluate our model's performance on the validation set. We use the `evaluate` library to load the MRPC metric (Accuracy and F1 score).
During evaluation, we:
1. Set the model to evaluation mode (`model.eval()`).
2. Disable gradient calculation (`torch.no_grad()`) to save memory and computation.
3. Accumulate predictions and references batch by batch.
4. Compute the final metrics at the end.

In [12]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.8480392156862745, 'f1': 0.8934707903780069}

## 10. Combining Training and Evaluation (Optional)

Here is how you would combine the initialization and the training loop into a single continuous block of code for clarity. This is just reproducing what we did above but in one cell.

In [13]:
from transformers import AutoModelForSequenceClassification, get_scheduler
from torch.optim import AdamW

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
optimizer = AdamW(model.parameters(), lr=3e-5)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/1377 [00:00<?, ?it/s]

## 11. Distributed Training with 🤗 Accelerate

For more advanced setups (multiple GPUs, TPUs, etc.), we can use the `Accelerate` library. `Accelerate` abstracts away the boilerplate code needed for distributed training.
Key changes:
- We initialize an `Accelerator`.
- We pass our dataloaders, model, and optimizer through `accelerator.prepare()`.
- We let the accelerator handle moving data to devices automatically.
- We replace `loss.backward()` with `accelerator.backward(loss)`.

In [14]:
from accelerate import Accelerator
from transformers import AutoModelForSequenceClassification, get_scheduler
from torch.optim import AdamW

accelerator = Accelerator()

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
optimizer = AdamW(model.parameters(), lr=3e-5)

train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_dataloader, eval_dataloader, model, optimizer
)

num_epochs = 3
num_training_steps = num_epochs * len(train_dl)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dl:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/1377 [00:00<?, ?it/s]

## 12. Running Training in a Notebook environment

If you want to run an accelerated training loop from within a Jupyter Notebook (e.g., Colab with TPUs), you need to encapsulate your training logic in a function. 
Here we fix the previous missing `training_function` by putting our Accelerate training loop inside a function and then passing it to `notebook_launcher`.

In [15]:
from accelerate import notebook_launcher

def training_function():
    # We encapsulate the accelerate training loop in this function
    from accelerate import Accelerator
    from transformers import AutoModelForSequenceClassification, get_scheduler
    from torch.optim import AdamW
    from tqdm.auto import tqdm
    
    accelerator = Accelerator()
    
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
    optimizer = AdamW(model.parameters(), lr=3e-5)
    
    train_dl, eval_dl, model_acc, optimizer_acc = accelerator.prepare(
        train_dataloader, eval_dataloader, model, optimizer
    )
    
    num_epochs = 3
    num_training_steps = num_epochs * len(train_dl)
    lr_scheduler = get_scheduler(
        "linear",
        optimizer=optimizer_acc,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )
    
    progress_bar = tqdm(range(num_training_steps))
    
    model_acc.train()
    for epoch in range(num_epochs):
        for batch in train_dl:
            outputs = model_acc(**batch)
            loss = outputs.loss
            accelerator.backward(loss)
            
            optimizer_acc.step()
            lr_scheduler.step()
            optimizer_acc.zero_grad()
            progress_bar.update(1)

# Launch the training in the notebook
notebook_launcher(training_function, num_processes=1)

NameError: name 'training_function' is not defined